# Convert Label Studio Export → BIO-Tagged NER Dataset

Takes your `label_studio_export.json` (character-span annotations) and
converts it into token-level BIO tags — the format your BiLSTM-CRF model
needs for training.

**Output:** `ner_bio_dataset.json` — a list of `{tokens: [...], tags: [...]}`
records, one per sentence, ready for Phase 2 preprocessing (vocab building,
label encoding, train/val split).


## 1. Setup

In [1]:
%pip install pandas -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import json

INPUT_JSON = "../data/annotated/label_studio_export.json"
OUTPUT_JSON = "../data/annotated/ner_bio_dataset.json"

with open(INPUT_JSON, encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} tasks.")

Loaded 700 tasks.


## 2. Tokenizer with character offsets

Simple but robust: splits on whitespace, then further splits off leading/
trailing punctuation as separate tokens (so "Prevacid," becomes two tokens:
"Prevacid" and ",", each keeping its own correct offset). This matters
because your labeled spans exclude punctuation, so the tokenizer needs to
match that boundary exactly or tokens will get mislabeled.

In [3]:
TOKEN_RE = re.compile(r"\w+(?:\.\w+)*|[^\w\s]")
# \w+(?:\.\w+)* keeps things like "b.i.d" or "0.25" as one token;
# [^\w\s] catches standalone punctuation (commas, periods) as separate tokens.

def tokenize(sentence):
    """Return list of (start, end, token_text)."""
    return [(m.start(), m.end(), m.group()) for m in TOKEN_RE.finditer(sentence)]

## 3. Assign BIO tags per token based on span overlap

For each token, find which labeled span (if any) it falls inside. The first
token of a span gets `B-LABEL`, subsequent tokens inside the same span get
`I-LABEL`, tokens outside any span get `O`.

In [4]:
def spans_to_bio(sentence, spans):
    """
    spans: list of dicts like {"start": int, "end": int, "labels": [str]}
    Returns (tokens, tags) — parallel lists.
    """
    tokens = tokenize(sentence)
    tags = ["O"] * len(tokens)

    # sort spans by start position so overlapping/adjacent spans resolve consistently
    sorted_spans = sorted(spans, key=lambda s: s["start"])

    for span in sorted_spans:
        s_start, s_end = span["start"], span["end"]
        label = span["labels"][0]
        first_token_in_span = True
        for i, (t_start, t_end, t_text) in enumerate(tokens):
            # token counts as "inside" the span if there's any overlap
            if t_start >= s_end or t_end <= s_start:
                continue
            if tags[i] != "O":
                continue  # already tagged by an earlier (higher-priority) span, skip
            tags[i] = f"B-{label}" if first_token_in_span else f"I-{label}"
            first_token_in_span = False

    token_texts = [t[2] for t in tokens]
    return token_texts, tags

## 4. Sanity check on one real example

In [5]:
example_task = tasks[0]
sentence = example_task["data"]["sentence"]
spans = [r["value"] for r in example_task["annotations"][-1]["result"]]

tok, tags = spans_to_bio(sentence, spans)
for t, g in zip(tok, tags):
    print(f"{t:15s} {g}")

Triglide        O
50              B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
Prevacid        B-DRUG
30              B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
Plavix          B-DRUG
75              B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
potassium       B-DRUG
10              B-DOSAGE
mEq             I-DOSAGE
daily           B-FREQUENCY
,               O
Lasix           B-DRUG
60              B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
folate          B-DRUG
1               B-DOSAGE
mg              I-DOSAGE
b.i.d           B-FREQUENCY
.               O
,               O
Niaspan         B-DRUG
500             B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
atenolol        B-DRUG
50              B-DOSAGE
mg              I-DOSAGE
daily           B-FREQUENCY
,               O
enalapril       B-DRUG
10          

## 5. Run over the full dataset

In [6]:
records = []
skipped = 0

for task in tasks:
    sentence = task["data"]["sentence"]
    anns = task.get("annotations", [])
    if not anns:
        skipped += 1
        continue

    results = anns[-1].get("result", [])
    if not results:
        skipped += 1
        continue

    spans = [r["value"] for r in results if r.get("type") == "labels"]
    if not spans:
        skipped += 1
        continue

    tokens, tags = spans_to_bio(sentence, spans)
    records.append({
        "sentence": sentence,
        "tokens": tokens,
        "tags": tags,
        "file": task["data"].get("file", ""),
        "specialty": task["data"].get("specialty", ""),
    })

print(f"Converted {len(records)} sentences to BIO format.")
print(f"Skipped {skipped} sentences (no annotation / no tagged spans).")

Converted 689 sentences to BIO format.
Skipped 11 sentences (no annotation / no tagged spans).


## 6. Dataset health check — entity counts per tag

In [7]:
from collections import Counter

tag_counts = Counter()
for rec in records:
    for tag in rec["tags"]:
        if tag != "O":
            entity = tag.split("-", 1)[1]
            tag_counts[entity] += 1

print("Entity instance counts (B- tags only, i.e. one count per entity mention):")
b_tag_counts = Counter()
for rec in records:
    for tag in rec["tags"]:
        if tag.startswith("B-"):
            b_tag_counts[tag[2:]] += 1

for k, v in b_tag_counts.most_common():
    print(f"  {k:12s} {v}")

print()
print("Total sentences:", len(records))
print("Avg tokens per sentence:", sum(len(r['tokens']) for r in records) / len(records))

Entity instance counts (B- tags only, i.e. one count per entity mention):
  DRUG         1698
  DOSAGE       1042
  FREQUENCY    785
  DIAGNOSIS    57
  DURATION     24

Total sentences: 689
Avg tokens per sentence: 16.872278664731496


## 7. Save the final BIO dataset

In [8]:
import os
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} BIO-tagged records -> {OUTPUT_JSON}")

Saved 689 BIO-tagged records -> ../data/annotated/ner_bio_dataset.json


## What's next

This `ner_bio_dataset.json` is your Phase 2 input. Next steps:
1. Build vocabulary (word → index) and label encoding (BIO tag → index)
2. Split into train/validation sets
3. Pad sequences to a fixed length
4. Build and train the BiLSTM-CRF model in TensorFlow/Keras
5. Evaluate F1 per entity class

If any entity type shows a very low count in the health check above (e.g.
SYMPTOM, since it was never auto-tagged), that's worth knowing now —
sparse categories are harder for the model to learn well, and you may want
to prioritize adding a few more manual SYMPTOM tags before training.
